# 💾 Manejo de Archivos — del CRUD en memoria al CRUD persistente
## Algoritmos y Estructuras de Datos I — UADE

---

Hasta ahora el CRUD del proyecto guarda los datos **en memoria**: cuando el programa termina, todo se pierde.

En este notebook vamos a aprender a **persistir** esos datos en archivos, de manera que sobrevivan entre ejecuciones.

### Hoja de ruta

| Bloque | Tema |
|---|---|
| 1 | Rutas: cómo Python encuentra los archivos |
| 2 | `open()`, modos y `with` — las bases |
| 3 | TXT — lectura y escritura simple |
| 4 | CSV — para listas de listas |
| 5 | JSON — para listas de diccionarios ⭐ |
| 6 | `read_data()` y `write_data()` — el puente con el CRUD |

> ⭐ El formato más relevante para el Proyecto Final: JSON mapea directamente a la estructura de lista de diccionarios que ya usamos.


---
## Bloque 1 — Rutas: cómo Python encuentra los archivos

Antes de abrir cualquier archivo hay que saber **dónde está parado Python** cuando ejecuta el script.

### Ruta absoluta vs ruta relativa

| Tipo | Descripción | Ejemplo |
|---|---|---|
| **Absoluta** | La ruta completa desde la raíz del sistema de archivos | `C:/Users/ana/proyecto/data/productos.json` |
| **Relativa** | Relativa al directorio de trabajo actual (`cwd`) | `data/productos.json` |

La ruta relativa es **preferible** porque el proyecto funciona en cualquier máquina sin modificar rutas.

### Directorio de trabajo actual (cwd)

Python siempre tiene un "directorio actual" desde donde resuelve las rutas relativas.  
Es el directorio desde donde ejecutamos el script, **no** donde está el script.


In [5]:
import os

# Ver el directorio de trabajo actual
print("Directorio actual (cwd):", os.getcwd())

# Construir rutas de forma segura con os.path.join
# Esto funciona en Windows, Mac y Linux sin cambios
ruta = os.path.join("data", "productos.json")
print("Ruta construida:", ruta)

# Verificar si un archivo o carpeta existe
print("¿Existe la carpeta 'data'?", os.path.exists("data"))

Directorio actual (cwd): c:\Users\JPy\Documents\UADE\UADE - Programacion I\1C2026\Progra_1_565944\Github_repo_oficial\Unidad_11
Ruta construida: data\productos.json
¿Existe la carpeta 'data'? True


En el archivo service.py les conviene usar este otro método

In [ ]:
from pathlib import Path
BASE_DIR = Path(__file__).resolve().parent
RUTA_PRODUCTOS = BASE_DIR.parent.parent / "data" / "productos.json"

### Estructura de carpetas recomendada para el proyecto

```
proyecto/
├── src/
│   ├── core/
│   │   ├── services.py
│   │   └── storage.py
│   ├── interface/
│   │   └── inputHandler.py
│   ├── shared/
│   │   └── utilities.py
│   └── app
|       └── main.py
├── test/
│   └── ...
└── data/
    ├── productos.json
    └── clientes.json
```

> ⚠️ Si ejecutás `python main.py` desde la carpeta `app/`, el cwd será `app/` y la ruta `data/productos.json` **no va a encontrar** la carpeta `data/` que está un nivel arriba.  
> Por eso en el proyecto se recomienda ejecutar desde la raíz: `python -m app.main`


In [ ]:
import os

# Crear la carpeta data/ si no existe (igual que "CREATE TABLE IF NOT EXISTS")
os.makedirs("data", exist_ok=True)
print("Carpeta 'data/' lista.")

# exist_ok=True evita el error si la carpeta ya existe

---
## Bloque 2 — `open()`, modos y `with`

### La función `open()`

```python
open(file, mode="r", encoding="utf-8")
```

| Parámetro | Descripción |
|---|---|
| `file` | Ruta al archivo (string) |
| `mode` | Modo de apertura (ver tabla abajo) |
| `encoding` | Codificación de caracteres. Siempre usar `"utf-8"` para soportar español |

### Modos de apertura

| Modo | Nombre | Crea si no existe | Borra contenido previo | Posición inicial |
|---|---|---|---|---|
| `"r"` | Solo lectura | ❌ Error | ❌ | Inicio |
| `"w"` | Escritura (sobrescribe) | ✅ | ✅ | Inicio |
| `"a"` | Append (agrega al final) | ✅ | ❌ | Final |
| `"x"` | Creación exclusiva | ✅ (falla si existe) | — | Inicio |

> 💡 Para el CRUD usamos principalmente `"r"` (leer) y `"w"` (escribir/reemplazar todo). El modo `"a"` es útil para logs.

### Por qué usar `with` en lugar de `open()` + `close()`


In [6]:
# ❌ Sin with: si ocurre un error entre open y close, el archivo queda abierto
# file = open("data/test.txt", "w", encoding="utf-8")
# file.write("algo")
# file.close()   ← puede no ejecutarse si hay un error antes

# ✅ Con with: Python garantiza el cierre aunque ocurra un error
with open("data/test.txt", "w", encoding="utf-8") as file:
    file.write("Hola desde with\n")

print("Archivo creado y cerrado correctamente.")
# Cuando termina el bloque with, file.close() se llama automáticamente

Archivo creado y cerrado correctamente.


---
## Bloque 3 — Archivos TXT

El formato más simple. Útil para logs, notas y texto libre.  
**Para datos estructurados (listas, diccionarios) preferimos CSV o JSON.**

### Crear y escribir


In [7]:
# Crear el archivo si no existe y escribir contenido
with open("data/log.txt", "w", encoding="utf-8") as file:
    file.write("=== Log del sistema ===\n")
    file.write("Producto agregado: Lápiz negro HB\n")
    file.write("Producto agregado: Bolígrafo azul\n")

print("Archivo log.txt creado.")

Archivo log.txt creado.


### Leer el contenido completo vs línea por línea


In [8]:
# Leer todo el contenido de una vez
with open("data/log.txt", "r", encoding="utf-8") as file:
    contenido = file.read()
    print("--- read() ---")
    print(contenido)

# Leer línea por línea (útil para archivos grandes)
with open("data/log.txt", "r", encoding="utf-8") as file:
    print("--- readlines() ---")
    for linea in file:
        print(linea.lower().strip())   # strip() elimina el \n al final

--- read() ---
=== Log del sistema ===
Producto agregado: Lápiz negro HB
Producto agregado: Bolígrafo azul

--- readlines() ---
=== log del sistema ===
producto agregado: lápiz negro hb
producto agregado: bolígrafo azul


In [ ]:
# Agregar una línea al final sin borrar el contenido (modo "a")
with open("data/log.txt", "a", encoding="utf-8") as file:
    file.write("Producto eliminado: id=1\n")

# Verificamos
with open("data/log.txt", "r", encoding="utf-8") as file:
    print(file.read())

### ¿Cuándo usar TXT?

✅ Logs de operaciones del sistema  
✅ Archivos de configuración simple  
❌ Listas de productos, clientes, ventas → usar **JSON**


---
## Bloque 4 — Archivos CSV

CSV (Comma Separated Values) es el formato estándar para **datos tabulares**: Excel, Google Sheets, pandas.  
Es ideal para **listas de listas**. Para listas de diccionarios, JSON es más natural (ver Bloque 5).

### El módulo `csv`

```python
import csv

csv.writer(file)          # escribe filas como listas
csv.reader(file)          # lee filas como listas
```

### Crear y escribir con headers


In [9]:
import csv

# Dataset: lista de listas
ventas = [
    [1, "2025-03-15", 1, 3, 1050.0],
    [2, "2025-03-16", 2, 1,  500.0],
    [3, "2025-03-17", 1, 2,  700.0],
]
headers = ["id_venta", "fecha", "id_cliente", "id_producto", "total"]

# Crear el archivo CSV con headers
with open("data/ventas.csv", "w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(headers)    # writerow: escribe UNA fila
    writer.writerows(ventas)    # writerows: escribe TODAS las filas de una vez

print("ventas.csv creado.")

ventas.csv creado.


### Parámetros importantes de `open()` para CSV

| Parámetro | Valor | Por qué |
|---|---|---|
| `newline=""` | Siempre vacío para CSV | Evita que Python agregue saltos de línea extra en Windows |
| `encoding` | `"utf-8"` | Soporta caracteres especiales |

### Leer un CSV


In [10]:
import csv

with open("data/ventas.csv", "r", newline="", encoding="utf-8") as file:
    reader = csv.reader(file)
    headers = next(reader)    # next() lee la primera fila (headers) y avanza el cursor
    print("Headers:", headers)
    print()
    for fila in reader:
        print(fila)

Headers: ['id_venta', 'fecha', 'id_cliente', 'id_producto', 'total']

['1', '2025-03-15', '1', '3', '1050.0']
['2', '2025-03-16', '2', '1', '500.0']
['3', '2025-03-17', '1', '2', '700.0']


### ⚠️ CSV solo almacena strings

Al leer un CSV, **todos los valores son strings**, aunque originalmente eran números.  
Hay que convertirlos manualmente:


In [ ]:
import csv

with open("data/ventas.csv", "r", newline="", encoding="utf-8") as file:
    reader = csv.reader(file)
    next(reader)  # saltar headers
    for fila in reader:
        id_venta   = int(fila[0])
        fecha      = fila[1]           # string, OK
        id_cliente = int(fila[2])
        total      = float(fila[4])
        print(f"Venta {id_venta} | Cliente {id_cliente} | Total: ${total:.2f}")

### ¿Cuándo usar CSV?

✅ Datos tabulares para exportar a Excel o Google Sheets  
✅ Listas de listas con tipos simples  
✅ Integración con pandas  
❌ Listas de diccionarios → requiere conversión manual de tipos → usar **JSON**


---
## Bloque 5 — Archivos JSON ⭐

JSON (JavaScript Object Notation) es el formato ideal para **listas de diccionarios**.  
Mapea directamente a la estructura que ya usamos en el CRUD del proyecto.

### Ventajas sobre CSV para el proyecto

| Característica | CSV | JSON |
|---|---|---|
| Estructura | Filas y columnas | Objetos anidados |
| Tipos de dato | Todo como string | Preserva int, float, bool, list |
| Conversión al leer | Manual | Automática |
| Legibilidad | Media | Alta |
| Uso en APIs | Poco | Estándar |

### El módulo `json`

```python
import json

json.dump(objeto, file, indent=4, ensure_ascii=False)   # escribe al archivo
json.load(file)                                          # lee del archivo
json.dumps(objeto)                                       # convierte a string
json.loads(string)                                       # parsea desde string
```


### Crear el archivo JSON si no existe

Al igual que "CREATE TABLE IF NOT EXISTS" en SQL, vamos a inicializar el archivo solo si no existe:


In [12]:
import json
import os

def inicializar_json(ruta, valor_inicial=None):
    """Crea el archivo JSON con un valor inicial si no existe."""
    if valor_inicial is None:
        valor_inicial = []
    if not os.path.exists(ruta):
        with open(ruta, "w", encoding="utf-8") as file:
            json.dump(valor_inicial, file, indent=4, ensure_ascii=False)
        print(f"Archivo creado: {ruta}")
    else:
        print(f"Archivo ya existe: {ruta}")

# Inicializamos los archivos del proyecto
os.makedirs("data", exist_ok=True)
inicializar_json("data/productos.json")
# inicializar_json("data/clientes.json")

Archivo creado: data/productos.json


### Escribir: `json.dump()`


In [13]:
import json

productos = [
    {"id_producto": 1, "categoria": "Escritura", "nombre": "Lápiz negro HB",  "precio_unitario": 350.0},
    {"id_producto": 2, "categoria": "Escritura", "nombre": "Bolígrafo azul",   "precio_unitario": 500.0},
    {"id_producto": 3, "categoria": "Papelería", "nombre": "Cuaderno A4",      "precio_unitario": 1200.0},
]

with open("data/productos.json", "w", encoding="utf-8") as file:
    json.dump(productos, file, indent=4, ensure_ascii=False)

print("productos.json guardado.")

productos.json guardado.


### Parámetros de `json.dump()`

| Parámetro | Descripción | Recomendación |
|---|---|---|
| `indent` | Espacios de indentación para legibilidad | `4` para lectura humana, `None` para producción |
| `ensure_ascii` | `True`: convierte ñ, á, etc. a `\uXXXX` | `False` para proyectos en español |

### Leer: `json.load()`


In [14]:
import json

with open("data/productos.json", "r", encoding="utf-8") as file:
    productos = json.load(file)

print(f"Productos cargados: {len(productos)}")
print()
for p in productos:
    # Los tipos se preservan: id_producto es int, precio_unitario es float
    print(f"  [{p['id_producto']}] {p['nombre']} — ${p['precio_unitario']:.2f}  (tipo: {type(p['precio_unitario']).__name__})")

Productos cargados: 3

  [1] Lápiz negro HB — $350.00  (tipo: float)
  [2] Bolígrafo azul — $500.00  (tipo: float)
  [3] Cuaderno A4 — $1200.00  (tipo: float)


### Trabajar en memoria y persistir

El flujo correcto del CRUD es:

```
Inicio del programa
    └── read_data()  → carga JSON → lista de dicts en memoria
         └── operaciones CRUD (agregar, buscar, modificar, eliminar)
              └── write_data()  → lista de dicts → JSON en disco
```

Nunca abrimos el archivo para cada operación individual: **cargamos todo al inicio, operamos en memoria, guardamos al final**.


In [ ]:
import json

# Simulación del flujo completo en memoria

# 1. Cargar al inicio
with open("data/productos.json", "r", encoding="utf-8") as file:
    productos = json.load(file)

print(f"Cargados {len(productos)} productos.")

# 2. Operar en memoria (CREATE)
nuevo = {"id_producto": 4, "categoria": "Papelería", "nombre": "Marcador permanente", "precio_unitario": 1200.0}
productos.append(nuevo)
print(f"Producto agregado. Total en memoria: {len(productos)}")

# 3. Persistir al finalizar
with open("data/productos.json", "w", encoding="utf-8") as file:
    json.dump(productos, file, indent=4, ensure_ascii=False)

print("Datos persistidos en disco.")

# 4. Verificar leyendo de nuevo
with open("data/productos.json", "r", encoding="utf-8") as file:
    verificacion = json.load(file)
print(f"Verificación: {len(verificacion)} productos en disco.")

---
## Bloque 6 — `read_data()` y `write_data()`: el puente con el CRUD ⭐

Estas dos funciones van en un módulo `storage.py` y son el único punto de contacto entre los servicios y el disco.  
`services.py` las llama para cargar y guardar datos; nunca abre archivos directamente.

```
services.py
  ├── get_producto_by_id()   →  lee de memoria (ya cargada con read_data)
  ├── crear_producto()   →  modifica memoria + llama write_data()
  └── eliminar_producto()    →  modifica memoria + llama write_data()

storage.py
  ├── read_data()            →  JSON → lista de dicts
  └── write_data()           →  lista de dicts → JSON
```


In [ ]:
import json
import os

# ── storage.py ────────────────────────────────────────────────────────────────

def read_data(ruta):
    """
    Lee un archivo JSON y retorna su contenido como lista de diccionarios.
    Si el archivo no existe, lo crea vacío y retorna [].

    Parámetros:
        ruta (str): ruta relativa al archivo JSON. Ej: 'data/productos.json'

    Retorna:
        list: lista de diccionarios con los registros.
    """
    os.makedirs(os.path.dirname(ruta), exist_ok=True)

    if not os.path.exists(ruta):
        # Comportamiento "CREATE TABLE IF NOT EXISTS":
        # si no existe el archivo, lo creamos vacío y retornamos []
        with open(ruta, "w", encoding="utf-8") as file:
            json.dump([], file)
        return []

    try:
        with open(ruta, "r", encoding="utf-8") as file:
            return json.load(file)
    except (json.JSONDecodeError, OSError) as e:
        print(f"Error al leer {ruta}: {e}")
        return []


def write_data(ruta, datos):
    """
    Escribe una lista de diccionarios en un archivo JSON.
    Sobrescribe el contenido previo (reemplaza todo el archivo).

    Parámetros:
        ruta  (str):  ruta relativa al archivo JSON. Ej: 'data/productos.json'
        datos (list): lista de diccionarios a persistir.

    Retorna:
        bool: True si se guardó correctamente, False si hubo error.
    """
    os.makedirs(os.path.dirname(ruta), exist_ok=True)

    try:
        with open(ruta, "w", encoding="utf-8") as file:
            json.dump(datos, file, indent=4, ensure_ascii=False)
        return True
    except OSError as e:
        print(f"Error al escribir {ruta}: {e}")
        return False


# ── Pruebas ───────────────────────────────────────────────────────────────────
RUTA_PRODUCTOS = "data/productos.json"

# read_data: carga el archivo (o lo crea si no existe)
productos = read_data(RUTA_PRODUCTOS)
print(f"read_data() → {len(productos)} productos cargados.")

# Operamos en memoria
productos.append({
    "id_producto":    len(productos) + 1,
    "categoria":      "Papelería",
    "nombre":         "Corrector líquido",
    "precio_unitario": 850.0
})

# write_data: persiste todo
ok = write_data(RUTA_PRODUCTOS, productos)
print(f"write_data() → {'OK' if ok else 'ERROR'}")

# Verificamos
productos_verificados = read_data(RUTA_PRODUCTOS)
print(f"Verificación: {len(productos_verificados)} productos en disco.")

### Cómo las usan los servicios

Con `read_data()` y `write_data()` los servicios quedan limpios y sin lógica de archivos:


In [ ]:
# ── services.py (con storage) ─────────────────────────────────────────────────

RUTA_PRODUCTOS = "data/productos.json"

def generar_id(registros):
    return max((r["id_producto"] for r in registros), default=0) + 1

def get_producto_by_id(id_producto):
    """READ: busca un producto por id. Retorna el dict o None."""
    productos = read_data(RUTA_PRODUCTOS)
    for p in productos:
        if p["id_producto"] == id_producto:
            return p
    return None

def crear_producto(producto):
    """CREATE: agrega un producto y persiste."""
    productos = read_data(RUTA_PRODUCTOS)
    producto["id_producto"] = generar_id(productos)
    productos.append(producto)
    return write_data(RUTA_PRODUCTOS, productos)

def update_producto(id_producto, cambios):
    """UPDATE: modifica campos de un producto existente y persiste."""
    productos = read_data(RUTA_PRODUCTOS)
    for p in productos:
        if p["id_producto"] == id_producto:
            p.update(cambios)
            return write_data(RUTA_PRODUCTOS, productos)
    return False   # no encontrado

def eliminar_producto(id_producto):
    """DELETE: elimina un producto por id y persiste."""
    productos = read_data(RUTA_PRODUCTOS)
    nuevos = [p for p in productos if p["id_producto"] != id_producto]
    if len(nuevos) == len(productos):
        return False   # no se encontró el id
    return write_data(RUTA_PRODUCTOS, nuevos)


# ── Demo CRUD completo ────────────────────────────────────────────────────────
print("=== CREATE ===")
crear_producto({"categoria": "Escritura", "nombre": "Lápiz negro HB",  "precio_unitario": 350.0})
crear_producto({"categoria": "Escritura", "nombre": "Bolígrafo azul",   "precio_unitario": 500.0})
crear_producto({"categoria": "Papelería", "nombre": "Cuaderno A4",      "precio_unitario": 1200.0})
print(f"Productos creados: {len(read_data(RUTA_PRODUCTOS))}")

print()
print("=== READ ===")
p = get_producto_by_id(2)
print(f"get_producto_by_id(2): {p}")

print()
print("=== UPDATE ===")
ok = update_producto(2, {"precio_unitario": 650.0})
print(f"update_producto(2, precio=650): {'OK' if ok else 'No encontrado'}")
print(f"Verificación: {get_producto_by_id(2)}")

print()
print("=== DELETE ===")
ok = eliminar_producto(1)
print(f"eliminar_producto(1): {'OK' if ok else 'No encontrado'}")
print(f"Productos restantes: {len(read_data(RUTA_PRODUCTOS))}")

---
## 🏋️ Ejercicios prácticos


### Ejercicio 1 — `read_data()` para clientes

Usando las funciones `read_data()` y `write_data()` definidas arriba, escribí las funciones de servicio para la entidad **Cliente**:
- `get_cliente_by_id(id_cliente)` → retorna el dict del cliente o `None`
- `persistir_cliente(cliente)` → agrega y persiste
- `update_cliente(id_cliente, cambios)` → modifica y persiste

Los clientes tienen: `id_cliente`, `nombre`, `email`, `telefono`.


In [ ]:
RUTA_CLIENTES = "data/clientes.json"

def generar_id_cliente(clientes):
    return max((c["id_cliente"] for c in clientes), default=0) + 1

def get_cliente_by_id(id_cliente):
    # Tu solución 👇
    pass

def persistir_cliente(cliente):
    # Tu solución 👇
    pass

def update_cliente(id_cliente, cambios):
    # Tu solución 👇
    pass

# Casos de prueba — no modificar
persistir_cliente({"nombre": "Ana García",  "email": "ana@gmail.com",    "telefono": "011-4321-5678"})
persistir_cliente({"nombre": "Carlos López","email": "carlos@yahoo.com", "telefono": "4567-8901"})

c = get_cliente_by_id(1)
print(f"get_cliente_by_id(1): {c}")

ok = update_cliente(1, {"email": "ana.garcia@gmail.com"})
print(f"update_cliente(1): {'OK' if ok else 'No encontrado'}")
print(f"Verificación: {get_cliente_by_id(1)}")

### Ejercicio 2 — Log de operaciones en TXT

Agregá una función `log_operacion(operacion, detalle)` que escriba en modo `"a"` en `data/log.txt` cada vez que se realice una operación CRUD.

El formato de cada línea debe ser:
```
[2025-03-15 10:32:01] CREATE | Producto agregado: Lápiz negro HB
[2025-03-15 10:32:05] DELETE | Producto eliminado: id=1
```

> 💡 Usá `from datetime import datetime` y `datetime.now().strftime("%Y-%m-%d %H:%M:%S")` para el timestamp.


In [ ]:
from datetime import datetime

def log_operacion(operacion, detalle):
    # Tu solución 👇
    pass

# Casos de prueba — no modificar
log_operacion("CREATE", "Producto agregado: Lápiz negro HB")
log_operacion("UPDATE", "Precio actualizado: id=2, nuevo precio=650.0")
log_operacion("DELETE", "Producto eliminado: id=1")

# Verificar el contenido del log
with open("data/log.txt", "r", encoding="utf-8") as f:
    print(f.read())

### 🏆 Ejercicio 3 — Desafío: exportar productos a CSV

Escribí una función `exportar_productos_csv(ruta_json, ruta_csv)` que:
1. Lea los productos desde el JSON con `read_data()`
2. Los escriba en un CSV con headers
3. Retorne la cantidad de registros exportados

Esto simula una funcionalidad real: exportar datos del sistema a Excel para un reporte.


In [ ]:
import csv

def exportar_productos_csv(ruta_json, ruta_csv):
    # Tu solución 👇
    pass

# Caso de prueba — no modificar
cantidad = exportar_productos_csv("data/productos.json", "data/productos_export.csv")
print(f"Exportados: {cantidad} productos")

# Verificar el CSV
with open("data/productos_export.csv", "r", encoding="utf-8") as f:
    print(f.read())

---
## 💬 Reflexión

Respondé en esta celda (doble clic para editar):

**1. ¿Por qué elegirías JSON sobre CSV para persistir la lista de productos del CRUD? ¿Y cuándo usarías CSV?**

> _Tu respuesta acá_

**2. ¿Qué ventaja tiene centralizar la lógica de archivos en `read_data()` y `write_data()` en lugar de usar `open()` directamente en cada servicio?**

> _Tu respuesta acá_

**3. ¿Qué ocurre si dos usuarios ejecutan el programa al mismo tiempo y ambos llaman a `write_data()` con datos diferentes?**

> _Tu respuesta acá_
